# Polars Fundamentals: Series, DataFrames, and Portable I/O

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

Polars is a blazingly fast DataFrame library implemented in Rust. Unlike legacy data structures that store tabular data as fragmented Python object pointers or contiguous row blocks in NumPy, Polars implements the **Apache Arrow Columnar Format** as its native in-memory model.

### Key Architectural Characteristics
1. **Contiguous Columnar Buffers:** Arrow chunked arrays align data contiguously in memory, maximizing CPU L1/L2 cache hit rates.
2. **SIMD Vectorization:** Native Rust kernels execute Single Instruction Multiple Data operations across column vectors.
3. **Multi-Threaded Work-Stealing:** Built upon Rayon, operations execute concurrently across all available CPU cores without Global Interpreter Lock (GIL) contention.
4. **Zero-Copy Slicing:** Slices and projections create lightweight view wrappers without copying underlying memory buffers.

### Learning Objectives
1. Construct and manipulate `pl.Series` across numeric, string, and temporal domains.
2. Leverage specialized namespaces: `.str` for text manipulation and `.dt` for datetime intervals.
3. Create, inspect, and profile `pl.DataFrame` objects.
4. Execute robust file I/O: parsing and persisting CSV and JSON formats with explicit schemas.


In [ ]:
from datetime import datetime
from pathlib import Path

import polars as pl

print(f"Polars version: {pl.__version__}")

## 1. The `pl.Series` Abstract Data Type

A `Series` is a strongly typed, 1-dimensional contiguous array of elements backed by Apache Arrow memory. Every Series possesses a name and a well-defined `polars.DataType`.


In [ ]:
# Numeric Series creation
s_num = pl.Series("measurements", [10, 25, 42, 68, 91])
print("Series:")
print(s_num)
print(f"Data type: {s_num.dtype} | Name: {s_num.name} | Length: {len(s_num)}")

### 1.1 Statistical Reductions and Aggregations

Polars Series provide efficient SIMD-accelerated aggregations:


In [ ]:
print(f"Minimum: {s_num.min()}")
print(f"Maximum: {s_num.max()}")
print(f"Sum: {s_num.sum()}")
print(f"Mean: {s_num.mean():.2f}")
print(f"Median: {s_num.median():.2f}")
print(f"Sample Standard Deviation: {s_num.std():.2f}")
print(f"90th Percentile: {s_num.quantile(0.90)}")

## 2. Specialized Series Namespaces: Strings and Datetimes

### 2.1 String Manipulation via `.str`

Text columns benefit from vectorized string operations without Python loop overhead:


In [ ]:
labels = pl.Series(
    "classes", ["polar bear", "arctic wolf", "polar fox", "grizzly bear", "polar penguin"]
)

# String transformations
replaced = labels.str.replace("polar", "subpolar")
uppercased = labels.str.to_uppercase()
contains_bear = labels.str.contains("bear")

print("Original:")
print(labels)
print("Replaced:")
print(replaced)
print("Contains 'bear':")
print(contains_bear)

### 2.2 Temporal Ranges and Date Math via `.dt`

Polars supports high-precision timestamp and date sequences:


In [ ]:
t_start = datetime(2026, 10, 20)
t_stop = datetime(2026, 10, 28)

# Generate a daily date range
calendar_series = pl.date_range(t_start, t_stop, interval="1d", eager=True)
print("Calendar series:")
print(calendar_series)

# Extract temporal components
print("Days of month:", calendar_series.dt.day().to_list())
print("Days of week (1=Mon, 7=Sun):", calendar_series.dt.weekday().to_list())

## 3. The `pl.DataFrame` Construct

A `DataFrame` is a two-dimensional tabular data structure composed of aligned `Series` columns of equal length.


In [ ]:
df = pl.DataFrame(
    {
        "product_id": [101, 102, 103, 104, 105],
        "category": ["Sensors", "Optics", "Sensors", "Compute", "Optics"],
        "unit_cost": [45.5, 120.0, 38.0, 850.0, 210.0],
        "quantity": [14, 5, 22, 2, 8],
        "active": [True, True, False, True, True],
    }
)

print("DataFrame structure:")
print(df)
print(f"Shape: {df.shape} (rows, cols)")
print(f"Columns: {df.columns}")
print(f"Schema: {df.schema}")

### 3.1 Inspecting DataFrames

Polars provides quick inspection methods: `head()`, `tail()`, and `describe()`.


In [ ]:
print("Top 2 rows:")
print(df.head(2))

print("Bottom 2 rows:")
print(df.tail(2))

print("Summary Statistics:")
print(df.describe())

## 4. High-Performance File I/O: CSV and JSON

Polars implements multi-threaded CSV parsers written in Rust that substantially outperform standard C-based parsers.

### 4.1 Writing and Reading CSV


In [ ]:
output_csv = Path("temp_products.csv")

# Write DataFrame to CSV
df.write_csv(output_csv)
print(f"Exported to {output_csv}. File size: {output_csv.stat().st_size} bytes")

# Read CSV with schema inference
df_csv = pl.read_csv(output_csv)
print("Imported CSV DataFrame:")
print(df_csv)

assert df_csv.shape == df.shape
print("CSV integrity confirmed.")

### 4.2 Writing and Reading JSON

Polars supports standard JSON array-of-objects serialization and NDJSON (newline-delimited JSON).


In [ ]:
output_json = Path("temp_products.json")

# Write to standard JSON
df.write_json(output_json)
print(f"Exported to {output_json}. File size: {output_json.stat().st_size} bytes")

# Read from JSON
df_json = pl.read_json(output_json)
print("Imported JSON DataFrame:")
print(df_json)

assert df_json.shape == df.shape
print("JSON integrity confirmed.")

# Cleanup temporary files
for path in [output_csv, output_json]:
    if path.exists():
        path.unlink()
print("Temporary scratch files cleaned up.")

## 5. Practice Challenge: Sensor Log Processing

**Exercise:** Given a simulated sensor log with timestamp strings, temperature readings, and device statuses:
1. Create a `pl.DataFrame` from raw records.
2. Convert timestamp strings to formal datetime datatypes.
3. Compute the overall temperature range (max - min) and mean temperature.
4. Cleanse rows where device status is inactive (`is_operational == False`).


In [ ]:
raw_sensor_data = {
    "timestamp": [
        "2026-10-26 08:00:00",
        "2026-10-26 09:00:00",
        "2026-10-26 10:00:00",
        "2026-10-26 11:00:00",
        "2026-10-26 12:00:00",
    ],
    "device_id": ["cam_01", "cam_02", "cam_01", "cam_02", "cam_01"],
    "temp_c": [22.4, 28.1, 23.0, 31.5, 24.2],
    "is_operational": [True, True, True, False, True],
}

sensor_df = pl.DataFrame(raw_sensor_data)

# Process sensor logs
processed_df = sensor_df.with_columns(
    pl.col("timestamp").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
).filter(pl.col("is_operational"))

print("Processed and Filtered Sensor Data:")
print(processed_df)

temp_min = processed_df["temp_c"].min()
temp_max = processed_df["temp_c"].max()
temp_range = round(temp_max - temp_min, 2)
temp_mean = round(processed_df["temp_c"].mean(), 2)

print(f"Operational Temperature: Mean = {temp_mean} °C, Range = {temp_range} °C")

assert processed_df.height == 4
assert temp_range == 9.1
print("All assertions passed!")